# NB_Purview_Copilot_BronzeToSilver
**Source:** `AI_Medallion.Bronze.purview_copilot_usage` (OneLake) **or** local CSV folder (this notebook only)  
**Targets (reads):** `okta_user_for_ai`, `hc_silver_historical`  
**Target:** `AI_Medallion.Silver.purview_copilot_usage` (incremental merge on `purview_copilot_usage_record_id`)  
**Run after:** `NB_OktaUserforAI_Bronze_To_Silver`, `NB_HCHistorical_Bronze_To_Silver`  

**Local CSV workaround:** set `PURVIEW_BRONZE_SOURCE = "local_csv"` and drop exports into `PURVIEW_BRONZE_CSV_DIR`. All loader logic is in this notebook (no Common/io).  
Export header (8 columns only): `RecordId,CreationDate,RecordType,Operation,UserId,AuditData,AssociatedAdminUnits,AssociatedAdminUnitsNames` — `FileName` from CSV path; `ExecutionDate` from `AuditData.CreationTime`. Filter **`CreationDate`** in transform.  
If a run fails mid-write, **restart the kernel** and delete `%USERPROFILE%\.fabric\merge_staging\_break\purview_copilot_bronze` (and `_silver`) before re-run.  
`ConnectionRefusedError` on `spark.read` = **dead Spark JVM** (usually after OOM). Restart kernel → settings → **bootstrap** → loader → read (do not skip bootstrap).

In [ ]:
# Workspace folder: C:\spark-dev\CoE_transformation_framework (see README)
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_LOCAL_TABLE_CACHE", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "25000")
os.environ.setdefault("MEDALLION_SCAN_RETRIES", "6")
os.environ.setdefault("MEDALLION_OBJECT_STORE_CONCURRENCY", "1")
# Purview local CSV: avoid JVM multi-part parquet under merge_staging/_break (corrupt reads on Windows)
os.environ["MEDALLION_STAGING_SPARK_PARQUET"] = "0"  # required: run this cell before bootstrap
os.environ.setdefault("MEDALLION_STAGING_COLLECT_ROWS", "25000")
os.environ.setdefault("PYSPARK_DRIVER_MEMORY", "8g")  # before bootstrap; raise if silver OOM
# After OOM / ConnectionRefusedError, uncomment for one run then comment again:
# os.environ["MEDALLION_FRESH_SPARK"] = "1"
# If you see break_lineage_local + Java heap space: git pull, restart kernel.

# --- Purview bronze source (this notebook only) ---
# "local_csv" = read CSV files you downloaded manually (no OneLake / medallion_cache for bronze)
# "onelake"   = read_table_broken_lineage (default pipeline path)
PURVIEW_BRONZE_SOURCE = "local_csv"
PURVIEW_BRONZE_CSV_DIR = Path.home() / "Downloads" / "purview_copilot_bronze"
# PURVIEW_BRONZE_CSV_DIR = Path(r"C:\data\purview_copilot_csv")
PURVIEW_BRONZE_CSV_GLOB = "*.csv"

# CreationDate window (applied in transform)
PURVIEW_BRONZE_CREATION_MONTH = None
# PURVIEW_BRONZE_CREATION_MONTH = "2026-09"
PURVIEW_BRONZE_FILENAMES = []


In [ ]:
%run ./Common/bootstrap


In [ ]:
# Workaround loader: local CSV → bronze DataFrame (Purview copilot only; stays in this notebook).
import shutil
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from Common import io as _medallion_io


def _purview_assert_spark_alive():
    if "spark" not in globals():
        raise RuntimeError(
            "Spark is not defined. Run the bootstrap cell (%run Common/bootstrap) before read."
        )
    try:
        spark.sparkContext.defaultParallelism()
    except Exception as exc:
        raise RuntimeError(
            "Spark JVM is not running (WinError 10061 / ConnectionRefused is common after OOM).\n"
            "Restart the Jupyter kernel, then run in order: settings → bootstrap → this cell → read bronze.\n"
            "Optionally set MEDALLION_FRESH_SPARK=1 in settings for one run after a crash."
        ) from exc


def purview_break_lineage_chunked(df, label: str):
    """Materialize to local parquet via driver chunks (stable for large AuditData on Windows)."""
    _medallion_io.apply_local_merge_spark_conf(spark)
    path = Path.home() / ".fabric" / "merge_staging" / "_break" / label
    print(f"purview_break_lineage_chunked: {label} -> {path}", flush=True)
    shutil.rmtree(path, ignore_errors=True)
    _medallion_io._write_df_via_local_iterator_chunks(df, path)
    return spark.read.parquet(str(path))

# Manual export shape (no FileName / ExecutionDate columns in the file)
_PURVIEW_CSV_EXPORT_COLUMNS = (
    "RecordId",
    "CreationDate",
    "RecordType",
    "Operation",
    "UserId",
    "AuditData",
    "AssociatedAdminUnits",
    "AssociatedAdminUnitsNames",
)

_PURVIEW_BRONZE_CANONICAL = (
    "AuditData",
    "RecordId",
    "CreationDate",
    "FileName",
    "ExecutionDate",
    "UserId",
    "RecordType",
    "Operation",
    "AssociatedAdminUnits",
    "AssociatedAdminUnitsNames",
)

_PURVIEW_CSV_RENAME = {
    "auditdata": "AuditData",
    "recordid": "RecordId",
    "creationdate": "CreationDate",
    "creation time": "CreationDate",
    "filename": "FileName",
    "executiondate": "ExecutionDate",
    "ingestiondate": "ExecutionDate",
    "userid": "UserId",
    "user id": "UserId",
    "recordtype": "RecordType",
    "operation": "Operation",
    "associatedadminunits": "AssociatedAdminUnits",
    "associatedadminunitsnames": "AssociatedAdminUnitsNames",
}


def _purview_csv_normalize_columns(df):
    out = df
    for col_name in list(out.columns):
        key = col_name.strip().lower().replace("_", "")
        target = _PURVIEW_CSV_RENAME.get(key) or _PURVIEW_CSV_RENAME.get(
            col_name.strip().lower()
        )
        if target and col_name != target:
            out = out.withColumnRenamed(col_name, target)
    return out


def load_purview_bronze_from_local_csv(
    csv_dir: Path,
    *,
    glob_pattern: str = "*.csv",
    break_label: str = "purview_copilot_bronze",
):
    """Read manually downloaded CSV exports into a bronze-shaped Spark DataFrame."""
    _purview_assert_spark_alive()
    csv_dir = Path(csv_dir).expanduser().resolve()
    if not csv_dir.is_dir():
        raise FileNotFoundError(
            f"PURVIEW_BRONZE_CSV_DIR is not a folder: {csv_dir}\n"
            "Create it and copy your Purview / M365 audit CSV exports there."
        )
    files = sorted(csv_dir.glob(glob_pattern))
    if not files:
        raise FileNotFoundError(
            f"No files matching {glob_pattern!r} under {csv_dir}"
        )
    print(
        f"purview bronze (local CSV): {len(files)} file(s) from {csv_dir}",
        flush=True,
    )
    for path in files[:20]:
        print(f"  - {path.name}", flush=True)
    if len(files) > 20:
        print(f"  ... and {len(files) - 20} more", flush=True)

    raw = (
        spark.read.option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .option("quote", '"')
        .csv([str(p) for p in files])
    )
    raw = _purview_csv_normalize_columns(raw)

    missing_export = [c for c in _PURVIEW_CSV_EXPORT_COLUMNS if c not in raw.columns]
    if missing_export:
        raise ValueError(
            "CSV header should be: RecordId,CreationDate,RecordType,Operation,"
            "UserId,AuditData,AssociatedAdminUnits,AssociatedAdminUnitsNames. "
            f"Missing: {missing_export}. Have: {raw.columns}"
        )

    raw = (
        raw.withColumn("CreationDate", F.trim(F.col("CreationDate")))
        .withColumn("UserId", F.trim(F.col("UserId")))
        .withColumn("RecordId", F.trim(F.col("RecordId")))
        .withColumn(
            "FileName",
            F.element_at(F.split(F.input_file_name(), r"[/\\\\]"), -1),
        )
        .withColumn(
            "ExecutionDate",
            F.get_json_object(F.col("AuditData"), "$.CreationTime"),
        )
    )

    missing = [c for c in ("AuditData", "RecordId", "CreationDate", "UserId") if c not in raw.columns]
    if missing:
        raise ValueError(
            "CSV is missing required bronze columns after rename: "
            f"{missing}. Have: {raw.columns}. "
            "Export should include AuditData (JSON), RecordId, CreationDate, UserId."
        )

    for optional in _PURVIEW_BRONZE_CANONICAL:
        if optional not in raw.columns:
            raw = raw.withColumn(optional, F.lit(None).cast(StringType()))

    bronze_df = raw.select(*[F.col(c) for c in _PURVIEW_BRONZE_CANONICAL])
    _row_count = bronze_df.count()
    print(f"purview bronze (local CSV): {_row_count} rows loaded", flush=True)
    return purview_break_lineage_chunked(bronze_df, break_label)


In [ ]:
_purview_assert_spark_alive()
if PURVIEW_BRONZE_SOURCE == "local_csv":
    bronze = load_purview_bronze_from_local_csv(
        PURVIEW_BRONZE_CSV_DIR,
        glob_pattern=PURVIEW_BRONZE_CSV_GLOB,
    )
elif PURVIEW_BRONZE_SOURCE == "onelake":
    bronze = read_table_broken_lineage(
        TABLE_PURVIEW_COPILOT_BRONZE,
        "purview_copilot_bronze",
    )
else:
    raise ValueError(
        f"PURVIEW_BRONZE_SOURCE must be 'local_csv' or 'onelake', got {PURVIEW_BRONZE_SOURCE!r}"
    )


In [ ]:
okta = read_table_broken_lineage(TABLE_OKTA_SILVER, "okta_silver")


In [ ]:
hc = read_table_broken_lineage(TABLE_HC_SILVER, "hc_silver")


In [ ]:
apply_local_merge_spark_conf(spark)

_creation_col = (
    PURVIEW_BRONZE_FILTER_DATE_COLUMN
    if PURVIEW_BRONZE_FILTER_DATE_COLUMN in bronze.columns
    else "creation_date"
)
_file_col = "FileName" if "FileName" in bronze.columns else "file_name"

if PURVIEW_BRONZE_CREATION_MONTH:
    _month_start = F.to_date(F.lit(f"{PURVIEW_BRONZE_CREATION_MONTH}-01"))
    _month_end = F.add_months(_month_start, 1)
    _creation_date = F.to_timestamp(F.trim(F.col(_creation_col))).cast("date")
    bronze = bronze.filter(
        (_creation_date >= _month_start) & (_creation_date < _month_end)
    )
    min_date = f"{PURVIEW_BRONZE_CREATION_MONTH}-01"
    print(
        f"purview bronze: CreationDate in {PURVIEW_BRONZE_CREATION_MONTH}",
        flush=True,
    )
else:
    min_date = os.environ.get(
        "MEDALLION_PURVIEW_MIN_CREATION_DATE", PURVIEW_COPILOT_MIN_CREATION_DATE
    )

if PURVIEW_BRONZE_FILENAMES:
    bronze = bronze.filter(F.col(_file_col).isin(PURVIEW_BRONZE_FILENAMES))
    print(
        f"purview bronze: FileName filter ({len(PURVIEW_BRONZE_FILENAMES)} file(s))",
        flush=True,
    )

print(f"purview_copilot_usage: creation_date >= {min_date}")

write_df = build_purview_copilot_usage_silver(
    bronze,
    okta,
    hc,
    min_creation_date=min_date,
)

write_df = purview_break_lineage_chunked(write_df, "purview_copilot_silver")
publish_merge_staging(write_df, TABLE_PURVIEW_COPILOT_SILVER)


In [ ]:
merge_incremental(TABLE_PURVIEW_COPILOT_SILVER, PURVIEW_COPILOT_MERGE_KEY)
